# CatBoost + 1-Minute Bars + Stock Execution

This notebook runs the live mean-reversion engine with:

- underlying signal data from IBKR stock bars
- completed `1min` signal bars
- CatBoost entry filter enabled
- regular stock orders through IBKR

The signal always comes from the underlying stock. In this example, execution also trades the stock.

## 1. Imports

Run this from the repo root. If you opened the notebook from `examples/`, the path setup below adds the repo root to `sys.path`.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, Stock

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import (
    active_orders,
    all_orders,
    build_execution_instrument,
    filled_orders,
    print_order_snapshot,
    recent_fills,
    strategy_open_trades,
)
from model_filters.catboost_live_model_filter import CatBoostLiveModelFilter
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


## 2. User Settings

Set the symbol, IBKR connection, and CatBoost model path. The CatBoost model must have been trained with feature columns matching `config.model["feature_cols"]`.

In [ ]:
SYMBOL = "META"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 101

CATBOOST_MODEL_PATH = "models/live/catboost/meta_catboost.cbm"
CATBOOST_PROB_THRESHOLD = 0.55

OPEN_TRADES_PATH = f"{SYMBOL}_stock_1min_catboost_open_trades.csv"


## 3. Build Config

Important pieces:

- `features.bar.type = "time"`
- `features.bar.timeframe = "1min"`
- `model.enabled = True`
- `execution.instrument.type = "stock"`

In [ ]:
raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["paths"]["open_trades_path"] = OPEN_TRADES_PATH
raw["paths"]["catboost_model_path"] = CATBOOST_MODEL_PATH

raw["features"]["bar"] = {
    "type": "time",
    "timeframe": "1min",
    "history_window": 390,
}

raw["model"]["enabled"] = True
raw["model"]["prob_threshold"] = CATBOOST_PROB_THRESHOLD

raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "SMART",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

config = LiveTradingConfig.from_dict(raw)
config.raw["features"]["bar"], config.raw["execution"]["instrument"], config.model["enabled"]


## 4. Connect IBKR and Subscribe to Underlying Bars

IBKR provides 5-second real-time bars. The strategy converts those into completed 1-minute signal bars internally.

In [ ]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock = Stock(SYMBOL, "SMART", "USD")
ib.qualifyContracts(stock)

real_time_bars = ib.reqRealTimeBars(
    stock,
    barSize=5,
    whatToShow="TRADES",
    useRTH=False,
)

print("Connected:", ib.isConnected())
print("Underlying:", stock)


## 5. Create Model Filter, Execution Instrument, and Strategy

`execution_instrument` resolves the tradable contract. Here it resolves to the same stock used for signals.

In [ ]:
model_filter = CatBoostLiveModelFilter(
    model_path=config.paths["catboost_model_path"],
    feature_cols=config.model["feature_cols"],
    prob_threshold=config.model.get("prob_threshold", 0.50),
    enabled=config.model.get("enabled", True),
)

execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

order_router = IBKRLimitOrderRouter(ib=ib, contract=stock)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
    execution_instrument=execution_instrument,
)

print("Execution instrument:", execution_instrument.instrument_type)
print("Loaded strategy open trades:", len(algo.open_trades))


## 6. Start / Stop Callback

Run the attach cell once. If you rerun setup, detach the old callback first to avoid duplicate orders.

In [ ]:
real_time_bars.updateEvent += algo.on_bar
print("Attached algo.on_bar")


In [ ]:
# Stop receiving callbacks from this algo instance.
real_time_bars.updateEvent -= algo.on_bar
print("Detached algo.on_bar")


## 7. Order and Trade Views

In [ ]:
print_order_snapshot(ib, algo)


In [ ]:
active_orders(ib)


In [ ]:
filled_orders(ib)


In [ ]:
recent_fills(ib)


In [ ]:
strategy_open_trades(algo)


In [ ]:
all_orders(ib)
